# UTH Computer Vision Coursework — Topic 11
## Notebook 02: Stable Diffusion Generation & Automated Curation Gate
- **Nhóm thực hiện**: Nhóm 4
- **Đề tài**: *Stable Diffusion-Based Synthetic Facial Dataset Generation for Deepfake Detection*
- **Mục tiêu**: Trực quan hóa 3 chế độ sinh ảnh giả bằng Stable Diffusion (, , ) và kiểm chứng bộ lọc chất lượng tự động .


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import torch

from src.common.metrics import compute_laplacian_sharpness
from src.pipelines.curate_image_dataset import ImageCurationGate


### 1. Nạp danh mục ảnh giả đã sinh ()

In [ ]:
curated_manifest = Path('data/02_synthetic_sd/manifests/curated_synthetic.jsonl')
if not curated_manifest.exists():
    curated_manifest = Path('data/processed/curated_manifest.jsonl')

curated_items = []
with open(curated_manifest, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            curated_items.append(json.loads(line))

print(f'Số lượng ảnh giả đạt chuẩn Curation Gate: {len(curated_items)}')


### 2. Kiểm định độ sắc nét Laplacian qua Curation Gate

In [ ]:
sharpness_scores = []
for item in curated_items:
    fp = item.get('file_path')
    if fp and Path(fp).exists():
        with Image.open(fp) as im:
            np_arr = np.array(im.convert('RGB'), dtype=np.float32) / 255.0
            t = torch.from_numpy(np_arr).permute(2, 0, 1)
            score = compute_laplacian_sharpness(t)
            sharpness_scores.append(score)

print(f'Độ sắc nét Laplacian trung bình: {np.mean(sharpness_scores):.2f}')
print(f'Giá trị nhỏ nhất / lớn nhất: {np.min(sharpness_scores):.2f} / {np.max(sharpness_scores):.2f}')

plt.figure(figsize=(7, 4))
plt.hist(sharpness_scores, bins=8, color='#9b59b6', edgecolor='black')
plt.axvline(x=10.0, color='r', linestyle='--', label='Ngưỡng lọc Curation Gate (min=10.0)')
plt.title('Phân Bố Độ Sắc Nét Laplacian Trên Tập Ảnh Giả')
plt.xlabel('Sharpness Score')
plt.ylabel('Số lượng ảnh')
plt.legend()
plt.tight_layout()
plt.show()


### 3. Trực quan hóa ảnh khuôn mặt giả sinh bởi Stable Diffusion

In [ ]:
fig, axes = plt.subplots(1, min(4, len(curated_items)), figsize=(12, 3.5))
if not isinstance(axes, np.ndarray):
    axes = np.array([axes])

for idx in range(min(4, len(curated_items))):
    item = curated_items[idx]
    fp = item.get('file_path')
    if fp and Path(fp).exists():
        im = Image.open(fp)
        axes[idx].imshow(im)
    axes[idx].set_title(item.get('manipulation_type', 'fake'), fontsize=10)
    axes[idx].axis('off')

plt.suptitle('Mẫu Khuôn Mặt Giả Sinh Bằng Stable Diffusion Đạt Chuẩn Curation Gate', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
